# Tugas Besar 2 IF3170 Inteligensi Artifisial

*Klasifikasi pembatalan pemesanan hotel (`is_canceled`).*

## IF3170_G08_Teazzi

- 13524034 - Jason Edward Salim
- 13524064 - Stefani Angeline Oroh
- 13524067 - Bryan Pratama Putra Hendra
- 13524079 - Angelina Andra Alanna
- 13524106 - Michael James Liman

## Table of Contents (CRISP-DM)

1. [**Business Understanding**](#1)
2. [**Data Understanding**](#2)
3. [**Data Preparation**](#3)
4. [**Modeling**](#4)
5. [**Evaluation**](#5)
6. [**Deployment**](#6)

## Initialization

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "src" else Path.cwd() / "src"))

from config import Settings, seed_everything
from eda import types, missing, duplicates, pie, bar, histogram, boxplot, iqr_outliers, target_rate, correlation
from preprocessing import load_data, split, drop_duplicates, Preprocessor, Pipeline
from validation import fit, holdout, kfold, compare, tune, best, importance, confusion, report, misclassified, submit
from models import KNN, LogisticRegression, SVM, DTL, BayesNetwork

seed_everything()

In [ ]:
train, test = load_data()
pd.Series({"train": len(train), "test": len(test)})

---

# Business Understanding <a name="1"></a>

---

Memprediksi apakah sebuah pemesanan hotel akan dibatalkan (`is_canceled = 1`) atau tidak (`is_canceled = 0`). Metrik evaluasi: macro F1-score.

> TODO: konteks bisnis dan dampak false positive / false negative.

---

# Data Understanding <a name="2"></a>

---

## Data Overview

In [ ]:
train

In [ ]:
display(types(train))
display(types(test))

In [ ]:
missing(train, test)

In [ ]:
duplicates(train, test)

## Target Distribution

In [ ]:
pie(train)
bar(train)

## Feature Analysis

In [ ]:
num_col = train.select_dtypes("number").columns.drop(Settings.TARGET)
cat_col = train.columns.difference(num_col).drop([Settings.ID, Settings.TARGET])

display(train[num_col].describe().T)
display(train[cat_col].describe().T)

In [ ]:
histogram(train, num_col)
boxplot(train, num_col)

In [ ]:
iqr_outliers(train, num_col)

In [ ]:
target_rate(train, list(cat_col) + ['special_requests', 'arrival_month', 'is_returning_guest'])

In [ ]:
correlation(train)

#### Insights

> TODO

---

# Data Preparation <a name="3"></a>

---

Data dibagi sebelum preprocessing. Seluruh langkah di-fit hanya pada `X_train`, lalu diterapkan ke `X_train`, `X_val`, dan `X_test`. Logika ada di `src/preprocessing.py` (fitur baru di `engineer`).

In [ ]:
X_train, X_val, y_train, y_val = split(train)
X_test = test.drop(columns=[Settings.ID])

X_train, y_train = drop_duplicates(X_train, y_train)
pd.Series({"X_train": len(X_train), "X_val": len(X_val), "X_test": len(X_test)})

In [ ]:
prep = Preprocessor(
    zero_as_missing=[],                                   # e.g. ["nightly_rate_idr"]
    num_strategy="median",
    cap=["days_before_arrival", "nightly_rate_idr"],
    ordinal={"meal_package": ["Room Only", "Breakfast", "Half Board", "Full Board"]},
    scaler="standard",                                    # 'standard' | 'minmax' | 'robust' | None
).fit(X_train)

X_train, X_val, X_test = prep.transform(X_train), prep.transform(X_val), prep.transform(X_test)
X_train.head()

#### Keputusan

> TODO: justifikasi imputasi, capping, encoding, scaling.

---

# Modeling <a name="4"></a>

---

Model from scratch ada di `src/models/` (implementasi `fit`/`predict`). Model sklearn dipakai sebagai pembanding. `METHOD` menentukan penanganan class imbalance (`None`, `'smote'`, `'undersample'`), hanya diterapkan pada data latih.

In [ ]:
METHOD = None

# Pipeline(model, select=k, pca=n) adds per-model feature selection and PCA

models = {
    "knn": KNN(k=5),
    "lr": LogisticRegression(lr=0.01, n_iter=1000),
    "svm": Pipeline(SVM(kernel="linear", C=0.1), select=10),
    "dtl": DTL(max_depth=None),
    "bn": BayesNetwork(),
}

In [ ]:
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

sk_models = {
    "sk_knn": KNeighborsClassifier(5),
    "sk_lr": SkLogisticRegression(max_iter=1000),
    "sk_svm": SVC(kernel="linear", C=1.0),
    "sk_dtl": DecisionTreeClassifier(max_depth=8, random_state=Settings.SEED),
    "sk_nb": GaussianNB(),
}

## SVM Feature Selection

SVM diselesaikan dalam bentuk dual dengan SMO, sehingga w = Σ αᵢyᵢxᵢ (hanya untuk kernel linear). Fitur diranking berdasarkan |w| SVM yang dilatih pada seluruh fitur, lalu `select` fitur teratas dipakai untuk melatih ulang SVM.

In [ ]:
svm_fitted = fit(models["svm"], X_train, y_train)
pd.Series(X_train.columns[svm_fitted.keep], name="svm selected features")

---

# Evaluation <a name="5"></a>

---

## Model Comparison

Holdout pada `X_val` dan K-Fold pada `X_train` dengan macro F1-score.

In [ ]:
all_models = {**models, **sk_models}
compare(holdout(all_models, X_train, y_train, X_val, y_val, METHOD), kfold(all_models, X_train, y_train, METHOD))

## Hyperparameter Tuning

In [ ]:
grids = {
    "knn": {"k": [5, 15, 31]},
    "lr": {"lr": [0.01, 0.1], "n_iter": [1000]},
    "svm": [
        {"kernel": ["linear"], "C": [0.1], "select": [10, None]},
        {"kernel": ["rbf"], "C": [1], "gamma": [0.1], "select": [None]},
    ],
    "dtl": {"max_depth": [5, 8, None]},
    "bn": {"n_bins": [3, 5, 10]},
}

grid_tables = {name: tune(models[name], grid, X_train, y_train, METHOD) for name, grid in grids.items()}
tuned_models = {name: best(models[name], table) for name, table in grid_tables.items()}
grid_tables["knn"].head()

In [ ]:
ranking = compare(
    holdout(tuned_models, X_train, y_train, X_val, y_val, METHOD),
    kfold(tuned_models, X_train, y_train, METHOD),
)
ranking

## Error Analysis

In [ ]:
# Pakai model from scratch terbaik, selama belum ada yang diimplementasikan, pakai baseline sklearn
if ranking.empty:
    ranking = compare(holdout(sk_models, X_train, y_train, X_val, y_val, METHOD), kfold(sk_models, X_train, y_train, METHOD))
    tuned_models = {**tuned_models, **sk_models}

best_name = ranking.index[0]
best_model = tuned_models[best_name]
best_name, best_model

In [ ]:
confusion(tuned_models, X_train, y_train, X_val, y_val, METHOD)

In [ ]:
report(tuned_models, X_train, y_train, X_val, y_val, METHOD)

In [ ]:
wrong_rows = misclassified(best_model, train, X_train, y_train, X_val, y_val, METHOD)
print(f"{len(wrong_rows)} dari {len(X_val)} baris salah")
wrong_rows["predicted"].map({1: "false positive", 0: "false negative"}).value_counts()

In [ ]:
importance(sk_models["sk_dtl"], X_train, y_train).head(10)

#### Insights

> TODO

---

# Deployment <a name="6"></a>

---

Model terbaik dilatih ulang dengan gabungan `X_train` dan `X_val`, lalu prediksi `X_test` disimpan ke `output/<model>_<timestamp>.csv` beserta `.pkl` modelnya.

In [ ]:
X_full, y_full = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])

submit(best_model, best_name, X_full, y_full, X_test, test[Settings.ID], METHOD)